*Author: Daria Andreeva, tg: @xufana*

# Семинар: оценивание агента — набор, разметка, траектории

Парный семинар к лекции «Как понять, что это работает, и работает хорошо?». Лекция была про то, как выбрать метрику, измеритель, интервал и решение. Здесь — то, что лекция оставила на практику: **как собрать набор**, **как выглядит разметка** и **как считать траектории агента своими руками**.

Весь семинар идёт на мок-магазине со стенда лекции «Трасса → граф»: заказы, четыре инструмента, задачи с эталонным графом. Агент не запускается. Трассы двух версий агента записаны заранее (120 штук), и среда переигрывает их заново при каждом запуске. Модель нужна только в разделе 4 (судья), но и он работает без GPU — на заранее записанных вердиктах.

| Что | Настоящее или сценарий |
|---|---|
| среда, наблюдения инструментов, конечное состояние, метрики графа, pass^k, каппа, bootstrap | считается здесь, по данным |
| шаги и тексты агента в трассах | сценарий: написаны руками под типовые ошибки агентов |
| готовые разметки `ann_1..3` и согласованная разметка | синтетические, с фиксированным seed — запасной вариант, если своей разметки нет |
| `judge_text_premade` — вердикты «судьи по тексту» | сценарий; настоящий судья — в разделе 4 на GPU |

**План**

0. Setup
1. Сборка набора: от контракта к категориям, запись эталона, тест эталона, синтетика, разделение и версия
2. Разметка трасс: инструкция, разметка в классе, согласие аннотаторов
3. Оценка траекторий: состояние, граф, автоматика против человека, pass^k
4. Судья для открытого текста: по тексту против по траектории, смещения
5. v1 против v2: шум против эффекта и решение
6. Регрессионный тест и бюджет оценивания
7. Что из этого идёт в design doc

Задания со звёздочкой (⭐) — для тех, кто хочет глубже.

## 0. Setup

In [ ]:
import sys, json, re, time, gc, hashlib, itertools, subprocess
from pathlib import Path

# Локально данные лежат рядом с ноутбуком, в Colab — клонируем репозиторий курса.
REPO = "https://github.com/xufana/llm-system-engineering.git"
if Path("shop/shop_env.py").exists():
    SHOP = Path("shop")
else:
    if not Path("llm-system-engineering").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    SHOP = Path("llm-system-engineering/06week_evaluation/shop")
sys.path.insert(0, str(SHOP.resolve()))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.metrics import cohen_kappa_score
from scipy.stats import bootstrap
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters

import shop_env as env

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)

TASKS = env.load_tasks()
TRACES = env.load_traces()
TR = {t["trace_id"]: t for t in TRACES}
print(f"задач: {len(TASKS)}, трасс: {len(TRACES)}")

## 1. Сборка набора

На лекции: «Без набора запросов нельзя отличить улучшение от регрессии». Здесь — как этот набор появляется. Порядок такой:

**контракт → категории задач → эталон каждой задачи → тест эталона → разделение и версия**

### 1.1 Контракт агента магазина

> Агент поддержки по запросу клиента сообщает статус заказа, отменяет заказ, если он ещё собирается, и оформляет возврат по доставленному заказу в пределах 14 дней. Инструменты: `get_order`, `cancel_order`, `create_return`, `finish`. Не делает того, о чём не просили. Если данных не хватает — уточняет. Если действие невозможно — объясняет почему. На вопросы не о заказах — вежливо отказывает. Не больше 8 шагов на задачу.

Каждая фраза контракта превращается в категорию задач и в проверку. Метрик без пункта контракта быть не должно — это требование КТ2.

| Пункт контракта | Категория задач | Чем проверяем |
|---|---|---|
| отменяет, оформляет возврат | действие; условие; несколько действий | состояние среды целиком |
| сообщает статус | информация | ключевые слова в ответе + граф (статус прочитан, а не придуман) |
| объясняет, почему невозможно | невозможное действие | состояние не изменилось + ключевые слова |
| не делает лишнего | все | состояние **целиком**, а не «есть нужный возврат»; лишние узлы графа |
| уточняет, если не хватает данных | неоднозначный запрос | состояние не изменилось + «номер» в ответе |
| вне заказов — отказ | вне домена | ключевые слова |
| переживает сбой инструмента | сбой инструмента | состояние + ретрай в графе |
| неверный номер заказа | ошибка во вводе | ключевые слова + граф |
| ≤ 8 шагов | все | флаг «завершён вызовом `finish`», циклы |

### 1.2 Набор задач

In [ ]:
tasks_df = pd.DataFrame([{
    "task_id": t["task_id"], "category": t["category"], "split": t["split"], "text": t["text"],
    "golden_nodes": len(t["golden"]["nodes"]), "faults": len(t.get("faults", [])),
} for t in TASKS.values()])
tasks_df

In [ ]:
tasks_df.groupby(["category", "split"]).size().unstack(fill_value=0)

Здесь по одной-две задачи на категорию: этого хватает, чтобы показать механику, но не чтобы мерить. В проекте размер задаётся так:

- **По категориям, а не «всего N».** В каждой категории — столько примеров, чтобы доля успеха по ней имела осмысленный интервал. На лекции: при n = 30 интервал доли ≈ ±16 п.п.
- **Два взвешивания.** Распределение реального трафика (как часто такое спрашивают) и распределение риска (во что обходится ошибка). Редкие, но дорогие категории — побочные действия, деньги — добирают сверх их доли в трафике. Итоговую метрику считают с весами трафика, а отчёт строят по категориям.
- **Ловушки обязательны.** Задачи, где правильное поведение — ничего не делать: невозможное действие, неоднозначный запрос, вне домена. Без них агент, который всегда что-то делает, выглядит идеальным.

### 1.3 Анатомия эталона

Одна запись набора — это не «вопрос и ответ», а спецификация проверки. Посмотрим на две задачи посложнее: с двумя независимыми действиями и со сбоем инструмента.

In [ ]:
for tid in ["T07", "T08"]:
    print(json.dumps(TASKS[tid], ensure_ascii=False, indent=1), "\n")

| Поле | Что задаёт | Какая проверка из него получается |
|---|---|---|
| `text` | запрос клиента | — |
| `expect` | **полное** ожидаемое состояние: какие статусы изменились, какие возвраты появились; всё остальное — как было | `check_state`: детерминированная, бесплатная |
| `final_mentions` | группы ключевых слов, которые должны быть в ответе (внутри группы — хотя бы одно) | `check_mentions`: грубая, но бесплатная |
| `golden.nodes` / `edges` | эталонный граф: какие вызовы нужны и какие должны идти раньше каких | node F1, выполнение зависимостей, точка расхождения |
| `golden.optional` | узлы, которые можно вызвать, а можно и нет | не считаются ни лишними, ни пропущенными |
| `golden_path` | одна каноническая последовательность | только для наивной метрики «соседних пар» (см. 3.3) |
| `faults` | сбои инструментов, которые среда подставит | сценарий «ошибка посередине» из КТ3 |
| `min_steps` | минимально возможное число шагов (с ретраем) | эффективность |
| `step_budget` | бюджет шагов | флаг «не завершён» |
| `split` | видимая или отложенная часть | см. 1.6 |

Два решения, которые стоит заметить:

- **`expect` — состояние целиком.** Проверка «возврат по 1042 есть» пропустит агента, который заодно отменил 1044. Проверка «состояние равно эталонному» — нет.
- **Граф — DAG, а не последовательность.** В T07 две независимые ветки, и правильно выполнить их можно в любом порядке. Эталон в виде одной цепочки наказал бы правильный путь.

In [ ]:
def draw_golden(task, ax):
    g = task["golden"]
    G = nx.DiGraph()
    G.add_nodes_from(g["nodes"])
    G.add_edges_from(g["edges"])
    for layer, nodes in enumerate(nx.topological_generations(G)):
        for n in nodes:
            G.nodes[n]["layer"] = layer
    pos = nx.multipartite_layout(G, subset_key="layer")
    opt = set(g.get("optional", []))
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=["#F5F5F5" if n in opt else "#EAF2FB" for n in G],
                           edgecolors="#2E6FBA", node_size=2600)
    nx.draw_networkx_labels(G, pos, {n: n.replace("(", "\n(") for n in G}, ax=ax, font_size=8)
    nx.draw_networkx_edges(G, pos, ax=ax, arrowsize=14, node_size=2600)
    ax.set_title(f"{task['task_id']}: {task['text']}", fontsize=9)
    ax.margins(0.15)
    ax.axis("off")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
draw_golden(TASKS["T07"], axes[0])
draw_golden(TASKS["T12"], axes[1])
plt.tight_layout()

### 1.4 Эталон — тоже код, и его тоже тестируют

Ошибка в эталоне хуже ошибки в агенте: агент делает правильно, а метрика пишет «провал», и вы чините то, что не сломано. Простейший тест: прогнать сам эталонный путь через среду. Он обязан пройти проверку состояния и не иметь точки расхождения.

In [ ]:
def steps_from_keys(keys, final_text="(эталонный ответ)"):
    steps = []
    for k in keys:
        if k == "finish":
            steps.append({"tool": "finish", "arg": final_text})
        else:
            tool, arg = re.fullmatch(r"(\w+)\((\d+)\)", k).groups()
            steps.append({"tool": tool, "arg": int(arg)})
    return steps


def test_golden(task):
    keys = list(task["golden_path"])
    for f in task.get("faults", []):  # на каждый сбой эталон делает ретрай
        i = keys.index(f["call"])
        keys[i:i] = [f["call"]] * f["times"]
    steps = steps_from_keys(keys)
    state, _ = env.replay(task, steps)
    return {"task_id": task["task_id"], "state_ok": env.check_state(task, state),
            "divergence": env.graph_metrics(task, steps)["divergence"]}

pd.DataFrame([test_golden(t) for t in TASKS.values()])

Теперь сломаем эталон так, как его обычно ломают руками: в T07 забыли записать ожидаемый возврат.

In [ ]:
import copy
broken = copy.deepcopy(TASKS["T07"])
broken["expect"] = {"status": {"1044": "отменён"}}  # забыли "returns": [1042]
test_golden(broken)

Тест эталона падает сразу, а не через неделю, когда кто-то заметит, что «агент стал хуже на T07». В проекте такой тест запускается на весь набор при каждом изменении набора.

### 1.5 Откуда брать задачи и как не сделать набор слишком лёгким

Три источника, по убыванию ценности:

1. **Реальные запросы** (логи, тикеты, переписка). Лучший источник, но у нового продукта его нет.
2. **Написанные руками по контракту.** По категориям из 1.1, с ловушками. Медленно, но язык живой.
3. **Синтетика.** Шаблоны или LLM. Быстро покрывает комбинации, но склонна быть слишком лёгкой и однообразной.

Сделаем синтетику по шаблонам. Эталон для неё выводится из правил среды, а не у модели: среда детерминирована, поэтому «что должно получиться» можно просто вычислить. По сути это rule-based оракул — и заодно baseline «не LLM», которого требует КТ2.

In [ ]:
TEMPLATES = {
    "return": "Оформите возврат по заказу {oid}.",
    "cancel": "Отмените заказ {oid}.",
    "status": "Какой статус у заказа {oid}?",
}
ACTION = {"return": "create_return", "cancel": "cancel_order"}


def synth_task(intent, oid, n):
    s = env.fresh_state()
    nodes = [f"get_order({oid})"]
    if intent in ACTION:
        try:
            env.TOOLS[ACTION[intent]](s, oid)        # оракул: что сделает правильное действие
            nodes.append(f"{ACTION[intent]}({oid})")
        except env.ToolError:
            pass                                     # действие невозможно → правильно ничего не менять
    nodes.append("finish")
    expect = {"returns": s["returns"],
              "status": {str(k): v["status"] for k, v in s["orders"].items() if v["status"] != env.ORDERS0[k]["status"]}}
    category = "информация" if intent == "status" else ("действие" if len(nodes) == 3 else "невозможное действие")
    return {"task_id": f"S{n:02d}", "category": category, "split": "visible", "text": TEMPLATES[intent].format(oid=oid),
            "expect": expect, "golden": {"nodes": nodes, "edges": [[a, b] for a, b in zip(nodes, nodes[1:])]},
            "golden_path": nodes, "step_budget": 8}


synth = [synth_task(i, oid, n) for n, (i, oid) in enumerate(itertools.product(TEMPLATES, env.ORDERS0))]
pd.DataFrame([{"task_id": t["task_id"], "category": t["category"], "text": t["text"], "golden": " → ".join(t["golden_path"])}
              for t in synth])

Комбинации состояний покрыты полностью (каждое намерение × каждый заказ), и эталон у каждой задачи правильный по построению. Но посмотрите на язык. Мера однообразия — среднее попарное сходство Жаккара по словам: 0 — тексты не пересекаются, 1 — одинаковые.

In [ ]:
def words(text):
    return set(re.findall(r"[а-яёa-z]+", text.lower())) - {"по", "у", "и", "в", "на"}

def mean_jaccard(texts):
    pairs = list(itertools.combinations([words(t) for t in texts], 2))
    return np.mean([len(a & b) / len(a | b) for a, b in pairs])

print(f"руками (12 задач):     {mean_jaccard([t['text'] for t in TASKS.values()]):.2f}")
print(f"по шаблону (15 задач): {mean_jaccard([t['text'] for t in synth]):.2f}")

Шаблонный набор проверяет логику («можно ли вернуть 1045»), но не язык. Реальный клиент напишет «верните деньги за кроссовки, номер где-то в письме» — и такой задачи в шаблонах нет. Рабочая смесь: шаблоны покрывают пространство состояний, руками и LLM-перефразами добирается язык, реальные запросы добавляются, как только появятся.

И главное предупреждение КТ2: набор, собранный **после** того, как система заработала, состоит из примеров, на которых она работает. Собирайте набор до.

**⭐ Задание:** перефразируйте шаблонные задачи локальной моделью (из раздела 4) так, чтобы смысл и номер заказа сохранились, и посчитайте сходство снова. Как проверить автоматически, что перефраз не поменял смысл? Подсказка: эталон у перефраза тот же, значит, ошибку перефраза можно поймать только по тексту.

### 1.6 Разделение и версия

- **Видимая часть** — на ней разрабатывают и смотрят ошибки.
- **Отложенная часть** — не открывается до финального замера (в проекте — до КТ4). Если на отложенной части смотрели ошибки и правили промпт, она стала видимой.
- **Версия** набора записывается рядом с каждым результатом вместе с версией агента, моделью и промптом. Число без версии набора — «число в чате».

Здесь T09–T11 отложены. На семинаре мы на них смотрим ради демонстрации; в проекте так не делают.

In [ ]:
DATASET_VERSION = hashlib.sha256((SHOP / "tasks.jsonl").read_bytes()).hexdigest()[:12]
RUN_META = {"dataset": f"tasks.jsonl@{DATASET_VERSION}", "env": "shop_env", "date": time.strftime("%Y-%m-%d")}
print(RUN_META)
print(tasks_df.split.value_counts().to_dict())

**⭐ Задание:** добавьте три задачи новой категории (например, «заказ уже отменён», «вернуть два заказа, один из которых вернуть нельзя») с полной записью эталона и прогоните их через `test_golden`. Какое поле эталона было сложнее всего заполнить?

## 2. Разметка трасс

Трасса — запись того, что агент делал: вызовы инструментов, наблюдения, финальный ответ. Автоматические проверки из раздела 1 смотрят на результат и путь. Человек нужен для двух вещей: проверить, что автоматика меряет то, что надо, и найти ошибки, которых она не видит. Сначала посмотрим, как выглядит трасса.

### 2.1 Как выглядит трасса

In [ ]:
def show_trace(trace_id):
    tr = TR[trace_id]
    task = TASKS[tr["task_id"]]
    print(f"{trace_id} · задача {task['task_id']} ({task['category']})")
    print(f"Клиент: {task['text']}\n")
    for i, st in enumerate(tr["steps"], 1):
        if st["tool"] == "finish":
            print(f"{i:>2}. finish → «{st['arg']}»")
        else:
            mark = "⚠ " if st["error"] else "  "
            print(f"{i:>2}. {st['tool']}({st['arg']})".ljust(26) + mark + st["obs"])
    if tr["steps"][-1]["tool"] != "finish":
        print(f"    … остановлен бюджетом шагов ({task['step_budget']})")
    print()

show_trace("T07-v1-r3")
show_trace("T06-v2-r2")

Обе трассы заканчиваются уверенным вежливым ответом. В первой возврата по 1042 нет: в журнале нет `create_return(1042)`. Во второй агент оформил возврат, о котором не просили. По тексту ответа ни то ни другое не видно.

### 2.2 Инструкция разметчика

Так выглядит разметка одной трассы: три поля и комментарий.

| Поле | Значения | Правило |
|---|---|---|
| `outcome` | `success` / `partial` / `fail` | **success** — среда в нужном состоянии и клиент получил верный ответ. **partial** — вреда нет и ответ честный, но задача не доведена, хотя могла быть. **fail** — неверное действие, неверный факт или ложное утверждение. |
| `first_error_step` | номер шага, с 1; пусто, если ошибок нет | Шаг, на котором агент **впервые** сделал не то. Если ошибка в пропуске шага — номер шага, где агент должен был его сделать. |
| `error_category` | одна категория из таксономии | Категория **первой** ошибки. Если ошибок нет — `none`. |

`outcome` — про результат, `error_category` — про путь. Трасса может быть `success` с категорией `extra_steps`: клиент получил своё, но агент сделал три лишних чтения.

**Таксономия ошибок**

| Категория | Что это | Пример |
|---|---|---|
| `none` | ошибок нет | — |
| `false_claim` | ответ утверждает действие, которого не было | «возврат оформлен», а `create_return` не вызван |
| `wrong_info` | сообщает факт, противоречащий наблюдению | инструмент: «в пути», ответ: «доставлен» |
| `wrong_action` | вызвал действие, которое по условию задачи вызывать не нужно (даже если среда его отклонила) | `create_return` по заказу «в пути» |
| `unrequested_side_effect` | изменил состояние, о чём не просили | просили отменить — оформил возврат |
| `missing_step` | пропустил обязательный шаг | действие без проверки заказа |
| `extra_steps` | лишние вызовы, результат верный | читает соседние заказы |
| `loop` | повторяет одно и то же без прогресса | `get_order` восемь раз подряд |
| `bad_error_handling` | неверно отреагировал на ошибку инструмента | сдался без ретрая; выдумал результат после ошибки |
| `should_clarify` | действовал наугад вместо уточнения | «отмените заказ» без номера → отменил какой-то |
| `out_of_scope` | ответил на вопрос вне домена | советует ноутбук |

Таксономия — часть эталона, и она дрейфует: по мере разметки появляются случаи, которые не ложатся ни в одну категорию (Shankar et al., 2024). Её версионируют, как промпт. В проекте таксономия своя, выведенная из вашего контракта.

### 2.3 Разметка в классе (≈10 минут)

1. Запустите `my_traces("ваше имя")` — получите свой блок трасс. Блоки перекрываются: каждую трассу размечают несколько человек, иначе согласие не посчитать.
2. Для каждой трассы: `show_trace(...)` → заполните форму (ссылку даст преподаватель). Одна трасса — одна отправка формы.
3. Пишите имя одинаково во всех отправках.

In [ ]:
LABEL_SET = sorted(t["trace_id"] for t in TRACES if t["label_set"])
N_BLOCKS = 4

def my_traces(name, n_blocks=N_BLOCKS):
    order = sorted(LABEL_SET, key=lambda t: hashlib.md5(t.encode()).hexdigest())
    blocks = [order[i::n_blocks] for i in range(n_blocks)]
    b = int(hashlib.md5(name.strip().lower().encode()).hexdigest(), 16) % n_blocks
    print(f"блок {b + 1} из {n_blocks}, трасс: {len(blocks[b])}")
    return sorted(blocks[b])

print(f"трасс в наборе для разметки: {len(LABEL_SET)}")
mine = my_traces("Иванова")
mine

In [ ]:
show_trace(mine[0])

### 2.4 Сбор разметки

Ответы формы попадают в Google Таблицу. Преподаватель публикует её как CSV («Файл → Поделиться → Опубликовать в интернете → CSV») и вставляет ссылку в `LABELS_URL`. Если ссылки нет, ноутбук берёт готовую разметку трёх синтетических аннотаторов, и дальше всё считается так же.

Готовые аннотаторы устроены по-разному — это пригодится в 2.5:
- `ann_1` — внимательный;
- `ann_2` — торопится: путает соседние категории и сбивается в номере шага;
- `ann_3` — читает в основном финальный ответ.

In [ ]:
LABELS_URL = ""  # ссылка на опубликованный CSV таблицы ответов формы

# заголовки вопросов формы → поля; сравнение по началу строки, чтобы пережить мелкие правки формы
FORM_FIELDS = {"отметка времени": "ts", "timestamp": "ts", "имя": "annotator", "id трассы": "trace_id",
               "итог": "outcome", "шаг": "first_error_step", "категория": "error_category", "комментарий": "comment"}


def load_labels(url=LABELS_URL):
    if not url:
        return pd.read_csv(SHOP / "labels_premade.csv"), "готовая разметка"
    raw = pd.read_csv(url)
    cols = {}
    for c in raw.columns:
        for prefix, field in FORM_FIELDS.items():
            if c.strip().lower().startswith(prefix):
                cols[c] = field
                break
    df = raw.rename(columns=cols)
    df["annotator"] = df["annotator"].astype(str).str.strip().str.lower()
    df["trace_id"] = df["trace_id"].astype(str).str.strip()
    for f in ["outcome", "error_category"]:  # варианты формы: "fail — ...", берём код до тире
        df[f] = df[f].astype(str).str.split("—").str[0].str.strip()
    df["first_error_step"] = pd.to_numeric(df["first_error_step"], errors="coerce")
    bad = ~df["trace_id"].isin(LABEL_SET)
    if bad.any():
        print(f"пропущено строк с неизвестным trace_id: {bad.sum()} → {sorted(df.loc[bad, 'trace_id'].unique())}")
    df = df[~bad]
    if "ts" in df:  # повторная отправка той же трассы — берём последнюю
        df = df.sort_values("ts")
    df = df.drop_duplicates(["annotator", "trace_id"], keep="last")
    return df[["annotator", "trace_id", "outcome", "first_error_step", "error_category"]], "разметка группы"


LABELS, source = load_labels()
print(f"{source}: {len(LABELS)} меток, аннотаторов: {LABELS.annotator.nunique()}")
LABELS.groupby("trace_id").size().describe()[["min", "mean", "max"]]

### 2.5 Согласие аннотаторов: это потолок метрики

Из лекции: если два асессора сходятся на κ = 0,6, требовать от судьи 0,9 бессмысленно. Посчитаем это для трёх полей разметки.

- **Попарная каппа Коэна** — для каждой пары аннотаторов на общих трассах.
- **Каппа Флейса** — обобщение на много аннотаторов сразу: одно число на поле. Нужна, когда трассу размечают разные люди, как в классе.

In [ ]:
def wide(field):
    return LABELS.pivot(index="trace_id", columns="annotator", values=field)


def pairwise_kappa(field, min_common=5):
    W = wide(field).astype(object).where(wide(field).notna(), None)
    rows = []
    for a, b in itertools.combinations(W.columns, 2):
        both = W[[a, b]].dropna()
        if len(both) >= min_common:
            rows.append({"a": a, "b": b, "общих": len(both),
                         "kappa": cohen_kappa_score(both[a].astype(str), both[b].astype(str))})
    return pd.DataFrame(rows)


def fleiss(field):
    W = wide(field).astype(object)
    m = W.notna().sum(axis=1)
    m_min = int(m[m >= 2].min())  # Флейс требует одинаковое число оценок на трассу
    rows = [r.dropna().astype(str).iloc[:m_min].tolist() for _, r in W[m >= 2].iterrows()]
    table, _ = aggregate_raters(np.array(rows, dtype=object))
    return fleiss_kappa(table)


LABELS["first_error_step"] = LABELS["first_error_step"].astype("Int64").astype(str).replace("<NA>", "—")
for field in ["outcome", "error_category", "first_error_step"]:
    print(f"{field:<17} Флейс κ = {fleiss(field):.2f}")
pairwise_kappa("error_category")

Посмотрим, где именно люди расходятся. Сначала матрица категорий для пары с самым низким согласием.

In [ ]:
pk = pairwise_kappa("error_category").sort_values("kappa")
a, b = pk.iloc[0][["a", "b"]]
W = wide("error_category")[[a, b]].dropna()
pd.crosstab(W[a], W[b], rownames=[a], colnames=[b])

In [ ]:
def disagreements(field="error_category"):
    W = wide(field)
    out = []
    for tid, r in W.iterrows():
        vals = r.dropna()
        if vals.nunique() > 1:
            out.append({"trace_id": tid, "меток": len(vals), "варианты": dict(vals.value_counts())})
    return pd.DataFrame(out).sort_values("меток", ascending=False)

disagreements()

Что обычно видно на готовой разметке (на своей проверьте сами):

- Внимательные аннотаторы расходятся на соседних категориях: `wrong_action` или `extra_steps`, `false_claim` или `missing_step`. Это не шум, а дыра в инструкции: правило «какую из двух категорий ставить» не прописано. Исправление — уточнить инструкцию и поднять её версию.
- `ann_3` ставит `success`/`none` там, где ответ звучит уверенно, — на ложных отчётах и побочных действиях. У судьи, который читает только финальный текст, будет то же слепое пятно (раздел 3.5 и 4).
- Расхождения обсуждают и сводят в **согласованную разметку** (adjudication). Здесь она лежит в `labels_adjudicated.csv` и дальше служит эталоном для сравнения автоматики.

Для КТ2 нужно ровно это: двойная разметка на подвыборке и посчитанное согласие.

**⭐ Задание:** допишите в инструкцию правило для самой спорной пары категорий, переразметьте 5 спорных трасс и посчитайте каппу заново.

## 3. Оценка траекторий

Четыре измерителя для одной траектории из лекции: конечное состояние, граф, судья по финальному тексту, судья по траектории. Первые два — детерминированные и бесплатные, считаем их для всех 120 трасс.

### 3.1 Конечное состояние

In [ ]:
ACTIONS = ("create_return", "cancel_order")


def side_effects(task, log):
    """Изменения состояния вызовами, которых нет в эталоне."""
    gold = set(task["golden"]["nodes"])
    return sum(1 for r in log if r["changed"] and r["key"] not in gold)


def eval_trace(tr):
    task = TASKS[tr["task_id"]]
    state, log = env.replay(task, tr["steps"])
    gm = env.graph_metrics(task, tr["steps"])
    div = gm["divergence"]
    return {"trace_id": tr["trace_id"], "task_id": tr["task_id"], "category": task["category"],
            "split": task["split"], "version": tr["agent_version"],
            "state_ok": env.check_state(task, state), "finished": state["finished"],
            "mentions_ok": env.check_mentions(task, state),
            "node_f1": gm["node_f1"], "dep_recall": gm["dep_recall"], "seq_edge_f1": gm["seq_edge_f1"],
            "efficiency": gm["efficiency"], "cycles": gm["cycles"], "steps": len(log),
            "side_effects": side_effects(task, log),
            "div_step": div[0] if div else None, "div_reason": div[1] if div else "",
            "judge_text": tr["judge_text_premade"]["score"], "label_set": tr["label_set"]}


E = pd.DataFrame([eval_trace(t) for t in TRACES])
E["success"] = E.state_ok & E.finished & E.mentions_ok
E[E.label_set][["trace_id", "category", "state_ok", "finished", "mentions_ok", "success", "side_effects"]]

Почему `finished` проверяется отдельно: в трассе с циклом (`T03`, агент восемь раз читает заказ) состояние **совпадает** с эталонным — ничего не должно было измениться, и ничего не изменилось. Проверка «состояние равно ожидаемому» поставила бы зачёт агенту, который так и не ответил клиенту.

In [ ]:
show_trace(E[(E.task_id == "T03") & (E.cycles > 0)].trace_id.iloc[0])

### 3.2 Граф траектории

Узлы — вызовы с аргументами, рёбра — порядок. Зелёные узлы есть в эталоне, оранжевые — лишние, серые снизу — пропущенные.

In [ ]:
def draw_trace(trace_id, ax):
    tr = TR[trace_id]
    task = TASKS[tr["task_id"]]
    _, log = env.replay(task, tr["steps"])
    keys = [r["key"] for r in log]
    gold = set(task["golden"]["nodes"])
    order = list(dict.fromkeys(keys))
    missing = [n for n in task["golden"]["nodes"] if n not in order and n not in task["golden"].get("optional", [])]
    counts = pd.Series(keys).value_counts()

    G = nx.DiGraph()
    pos = {k: (i, 0) for i, k in enumerate(order)}
    pos.update({k: (j + 0.5, -1) for j, k in enumerate(missing)})
    G.add_nodes_from(order + missing)
    edges = [(a, b) for a, b in zip(keys, keys[1:]) if a != b]
    G.add_edges_from(edges)
    colors = ["#3C9A5C" if k in gold else "#E8833A" for k in order] + ["#CCCCCC"] * len(missing)
    labels = {k: k.replace("(", "\n(") + (f"\n×{counts[k]}" if counts.get(k, 0) > 1 else "") for k in order + missing}
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colors, node_size=2300, alpha=0.85)
    nx.draw_networkx_labels(G, pos, labels, ax=ax, font_size=7)
    nx.draw_networkx_edges(G, pos, ax=ax, edgelist=edges, arrowsize=14, node_size=2300,
                           connectionstyle="arc3,rad=0.25")
    div = env.graph_metrics(task, tr["steps"])["divergence"]
    ax.set_title(f"{trace_id}: " + (f"расхождение на шаге {div[0]} — {div[1]}" if div else "совпадает с эталоном"), fontsize=9)
    ax.set_ylim(-1.6, 0.6)
    ax.axis("off")


fig, axes = plt.subplots(3, 1, figsize=(11, 9))
for ax, tid in zip(axes, ["T02-v2-r4", "T07-v1-r3", "T08-v1-r4"]):
    draw_trace(tid, ax)
plt.tight_layout()

- `T02-v2-r4` — результат верный, путь длинный: три лишних чтения и повтор. Проверка состояния ставит зачёт, граф показывает эффективность 0,5.
- `T07-v1-r3` — одна ветка из двух не выполнена. Это видно по пропущенным узлам без всякого чтения ответа.
- `T08-v1-r4` — сбой инструмента и ретрай. Повтор после **ошибки** не считается циклом: это правильное поведение.

### 3.3 Метрики графа и почему эталон — DAG

| Метрика | Что меряет |
|---|---|
| `node_f1` | те ли вызовы с теми ли аргументами (множества узлов) |
| `dep_recall` | доля зависимостей эталона, выполненных в правильном порядке |
| `seq_edge_f1` | наивно: совпадение соседних пар с одной канонической цепочкой (как на стенде лекции) |
| `efficiency` | `min_steps / шагов`: меньше 1 — лишние шаги, больше 1 — шаг пропущен |
| `cycles` | повтор успешного вызова без изменения состояния |
| `div_step`, `div_reason` | первый шаг, где трасса ушла от эталона, и почему |

In [ ]:
cols = ["trace_id", "success", "node_f1", "dep_recall", "seq_edge_f1", "efficiency", "cycles", "div_step", "div_reason"]
E[E.task_id.isin(["T07", "T12"]) & E.success][cols].drop_duplicates(subset=["node_f1", "seq_edge_f1", "div_step"])

Правильные трассы T07 и T12 с другим порядком веток получают `seq_edge_f1` заметно ниже 1, хотя ошибки нет. Метрика по зависимостям DAG (`dep_recall`) и точка расхождения их не штрафуют. Если у задачи несколько правильных путей, эталон должен это знать — так устроен WorfBench.

### 3.4 Автоматика против человека

Сравним детерминированные проверки с согласованной разметкой. Два вопроса: совпадает ли результат (`success` против `outcome`) и находит ли граф ту же первую ошибку, что человек.

In [ ]:
ADJ = pd.read_csv(SHOP / "labels_adjudicated.csv")
M = E[E.label_set].merge(ADJ, on="trace_id")
M["human_success"] = M.outcome == "success"
M["human_path_issue"] = M.error_category != "none"
M["auto_path_issue"] = M.div_step.notna()

display(pd.crosstab(M.success, M.outcome, rownames=["авто: success"], colnames=["человек: outcome"]))
display(pd.crosstab(M.auto_path_issue, M.human_path_issue, rownames=["граф: есть расхождение"],
                    colnames=["человек: есть ошибка пути"]))
both = M[M.auto_path_issue & M.human_path_issue]
print(f"шаг первой ошибки совпал у графа и человека: {(both.div_step == both.first_error_step).mean():.0%} из {len(both)}")

In [ ]:
show_cols = ["trace_id", "success", "outcome", "error_category", "first_error_step", "div_step", "div_reason", "mentions_ok"]
print("Человек видит ошибку, граф — нет:")
display(M[M.human_path_issue & ~M.auto_path_issue][show_cols])
print("Результат верный, путь — нет:")
display(M[M.success & M.human_path_issue][show_cols])

Разберите таблицы:

- **Граф не видит, человек видит.** Неверный факт в ответе (`wrong_info`), выдуманный результат после ошибки инструмента: путь правильный, ошибка в тексте. Часть таких случаев ловит `mentions_ok`, остальные — задача судьи по траектории (раздел 4). Отдельно посмотрите на «слепой» ретрай в T08: агент после таймаута сразу оформил возврат, не перечитав заказ. Граф считает чтение выполненным, потому что вызов был, хоть и с ошибкой. Это решение о том, как записан эталон, и его можно оспорить.
- **Результат верный, путь нет.** Действие без проверки, лишние чтения, попытка запрещённого действия, которую отклонила среда. Клиент ничего не заметил, но на тысяче запросов это стоимость, задержка и риск.

Вывод из лекции на конкретных числах: результат и путь — разные метрики. Нужны обе.

### 3.5 Судья по тексту против состояния

У каждой трассы есть вердикт «судьи по тексту» (1–5): он видит запрос клиента и финальный ответ, но не среду.

In [ ]:
E["judge_pass"] = E.judge_text >= 4
display(pd.crosstab(E.judge_pass, E.success, rownames=["судья по тексту ≥ 4"], colnames=["успех по состоянию"]))
fooled = E[E.judge_pass & ~E.success]
print(f"судья поставил ≥ 4 провальным трассам: {len(fooled)} из {(~E.success).sum()}")
fooled.drop_duplicates("task_id")[["trace_id", "category", "judge_text", "state_ok", "mentions_ok"]]

In [ ]:
show_trace("T01-v1-r3")
print("вердикт судьи:", TR["T01-v1-r3"]["judge_text_premade"])

Ложный отчёт о возврате получает от судьи 5 из 5 — «ответ полный, вежливый, называет срок». У судьи по тексту нет среды, и никакой rubric этого не исправит. Вопрос из лекции «агент пишет: возврат оформлен — какие три проверки скажут, правда ли это?» здесь закрывается кодом: состояние, узел в графе, лог инструмента.

### 3.6 pass@k и pass^k

У каждой задачи по 5 прогонов на версию. pass@k — хотя бы одна из k попыток успешна. pass^k — все k успешны: пользователь получает одну попытку, и она должна сработать. Обе величины считаются несмещённо по n = 5 прогонам и c успешным.

In [ ]:
succ = E.groupby(["task_id", "version"]).success.sum().unstack()
K = range(1, 6)
curves = {(v, name): [np.mean([f(5, c, k) for c in succ[v]]) for k in K]
          for v in ["v1", "v2"] for name, f in [("pass@k", env.pass_at_k), ("pass^k", env.pass_hat_k)]}

fig, ax = plt.subplots(figsize=(7, 4))
for (v, name), ys in curves.items():
    ax.plot(list(K), ys, marker="o", ls="-" if name == "pass^k" else "--",
            color="#2E6FBA" if v == "v1" else "#D9534F", label=f"{v} {name}")
ax.set_xlabel("k"); ax.set_ylabel("доля задач"); ax.set_ylim(0, 1.05); ax.grid(alpha=0.3); ax.legend()
ax.set_title("pass@k растёт, pass^k падает")
plt.tight_layout()
succ.T

pass@k у обеих версий быстро уходит к 1: «иногда получается» почти всё. pass^k разводит версии гораздо сильнее, чем доля успеха за одну попытку. Для продукта, где у клиента одна попытка, мерить надо pass^k — и бюджет оценивания умножается на k прогонов на задачу (раздел 6).

## 4. Судья для открытого текста

Детерминированные проверки покрыли действия. Остаётся финальный текст: верно ли он передаёт то, что произошло, честно ли объясняет отказ. Здесь нужен судья, и ему, по лекции, доверяют после проверки, а не по умолчанию.

План раздела:

1. Один судья в двух режимах: **по тексту** (видит запрос и ответ) и **по траектории** (видит ещё и журнал инструментов). Согласие каждого с человеком.
2. Второй судья другого семейства: согласие судей между собой.
3. Смещение по позиции: pairwise в двух порядках.
4. Смещение по длине: ответы, раздутые водой.

Rubric — одна ось и шкала с якорями. Одна ось на вызов снижает разброс; правила для граничных случаев прописаны явно, иначе судья решит их сам и каждый раз по-разному.

**Без GPU** раздел работает частично: считается согласие с человеком для заранее записанных вердиктов, а эксперименты со смещениями пропускаются. В Colab включите GPU: Runtime → Change runtime type → T4.

In [ ]:
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    HAS_GPU = False
USE_LOCAL_JUDGE = HAS_GPU

# две модели разных семейств, обе помещаются на T4 в fp16; грузятся по очереди
JUDGE_MODELS = {"qwen": "Qwen/Qwen2.5-3B-Instruct", "phi": "microsoft/Phi-4-mini-instruct"}

CACHE_PATH = Path("judge_cache.json")
CACHE = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}
JUDGE_STATS = {"calls": 0, "seconds": 0.0}
JUDGES = {}
print("локальный судья:", "включён" if USE_LOCAL_JUDGE else "выключен (нет GPU) — работаем на готовых вердиктах")

In [ ]:
def get_judge(name):
    """Держим в памяти одну модель: перед загрузкой новой выгружаем старую."""
    if name not in JUDGES:
        from transformers import pipeline
        for other in list(JUDGES):
            del JUDGES[other]
        gc.collect(); torch.cuda.empty_cache()
        pipe = pipeline("text-generation", model=JUDGE_MODELS[name], torch_dtype=torch.float16, device_map="auto")
        pipe.tokenizer.padding_side = "left"
        if pipe.tokenizer.pad_token is None:
            pipe.tokenizer.pad_token = pipe.tokenizer.eos_token
        JUDGES[name] = pipe
    return JUDGES[name]


def generate(name, prompts, temperature=0.0, seed=0, max_new_tokens=200):
    """Вызовы судьи с кэшем на диске: повторный запуск ячейки бесплатен, прогон воспроизводим."""
    keys = [hashlib.sha256(json.dumps([JUDGE_MODELS[name], p, temperature, seed], ensure_ascii=False).encode()).hexdigest()
            for p in prompts]
    todo = [(k, p) for k, p in zip(keys, prompts) if k not in CACHE]
    if todo:
        pipe = get_judge(name)
        kw = dict(max_new_tokens=max_new_tokens, return_full_text=False, batch_size=8, do_sample=temperature > 0)
        if temperature > 0:
            kw["temperature"] = temperature
            torch.manual_seed(seed)
        t0 = time.time()
        outs = pipe([[{"role": "user", "content": p}] for _, p in todo], **kw)
        JUDGE_STATS["seconds"] += time.time() - t0
        JUDGE_STATS["calls"] += len(todo)
        for (k, _), o in zip(todo, outs):
            CACHE[k] = o[0]["generated_text"]
        CACHE_PATH.write_text(json.dumps(CACHE, ensure_ascii=False))
    return [CACHE[k] for k in keys]

In [ ]:
from pydantic import BaseModel, Field, ValidationError


class Verdict(BaseModel):
    reasoning: str = Field(description="1–2 предложения: что в ответе подтверждено, что нет")
    score: int = Field(ge=1, le=5)


def parse_verdict(text):
    """Маленькие модели ломают JSON — доля непарсящихся ответов тоже метрика судьи."""
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            return Verdict.model_validate_json(m.group(0))
        except ValidationError:
            pass
    m = re.search(r'score"?\s*[:=]\s*([1-5])', text)
    return Verdict(reasoning="(не JSON)", score=int(m.group(1))) if m else None


RUBRIC = """Оцени финальный ответ агента поддержки интернет-магазина по шкале 1–5.
5 — ответ верен и полон: всё, что агент утверждает, {basis}; просьба клиента выполнена или честно объяснено, почему её нельзя выполнить.
4 — ответ верен, но неполон: нет срока, нет альтернативы, неточная формулировка.
3 — ответ честный, но задача клиента не решена, хотя могла быть решена.
2 — основная часть верна, но есть неподтверждённое утверждение.
1 — ответ ложный: утверждает действие, которого не было, или сообщает неверный факт; или ответа нет.
Граничные случаи: если агент сделал то, о чём клиент не просил, — не выше 2. Если вопрос не о заказах, правильный ответ — вежливый отказ, это 5."""

BASIS = {"text": "правдоподобно и отвечает на запрос (журнал действий агента тебе не показан)",
         "traj": "подтверждено журналом вызовов инструментов"}

JUDGE_TMPL = """Ты — строгий асессор качества ответов службы поддержки.
{rubric}

Запрос клиента: {request}
{log}Финальный ответ агента: {answer}

Сначала кратко порассуждай (1–2 предложения), затем выставь оценку.
Верни только JSON: {{"reasoning": "...", "score": N}}"""


def final_answer(tr):
    last = tr["steps"][-1]
    return last["arg"] if last["tool"] == "finish" else "(агент не дал финального ответа)"


def judge_prompt(tr, mode, answer=None):
    log = ""
    if mode == "traj":
        lines = [f"{i}. {s['tool']}({s['arg']}) → {s['obs']}" for i, s in enumerate(tr["steps"], 1) if s["tool"] != "finish"]
        log = "Журнал вызовов инструментов:\n" + ("\n".join(lines) or "(инструменты не вызывались)") + "\n"
    return JUDGE_TMPL.format(rubric=RUBRIC.format(basis=BASIS[mode]), request=TASKS[tr["task_id"]]["text"],
                             log=log, answer=answer or final_answer(tr))


print(judge_prompt(TR["T01-v1-r3"], "traj"))

### 4.1 По тексту против по траектории: согласие с человеком

Сравниваем бинарно: судья «за» — оценка ≥ 4; человек «за» — `outcome == success` в согласованной разметке. Строка `premade` — заранее записанные вердикты судьи по тексту, она считается и без GPU.

In [ ]:
LS = [TR[t] for t in LABEL_SET]
H = ADJ.set_index("trace_id").loc[LABEL_SET]
human_pass = (H.outcome == "success").to_numpy()

J = pd.DataFrame({"trace_id": LABEL_SET, "premade": [t["judge_text_premade"]["score"] for t in LS]})
if USE_LOCAL_JUDGE:
    for mode in ["text", "traj"]:
        verdicts = [parse_verdict(o) for o in generate("qwen", [judge_prompt(t, mode) for t in LS])]
        J[f"qwen_{mode}"] = [v.score if v else np.nan for v in verdicts]
        print(f"qwen_{mode}: не распарсилось {sum(v is None for v in verdicts)} из {len(verdicts)}")


def agreement_with_human(J, cols):
    rows = []
    for c in cols:
        ok = J[c].notna().to_numpy()
        jp = (J[c] >= 4).to_numpy()
        rows.append({"судья": c, "n": int(ok.sum()), "accuracy": (jp[ok] == human_pass[ok]).mean(),
                     "kappa": cohen_kappa_score(jp[ok], human_pass[ok]),
                     "пропустил провалов": int((jp & ~human_pass & ok).sum())})
    return pd.DataFrame(rows)


agreement_with_human(J, [c for c in J.columns if c != "trace_id"])

Сравните строки с потолком из 2.5 (согласие людей по `outcome`). Чего ожидать:

- судья по тексту ошибается ровно там, где ошибался `ann_3`: на уверенных ложных отчётах;
- журнал инструментов в промпте даёт судье шанс увидеть, что `create_return` не вызывался; насколько маленькая модель этим шансом пользуется — вопрос к вашим числам;
- столбец «пропустил провалов» для продукта важнее каппы: это ложные «всё хорошо».

### 4.2 Второй судья другого семейства

Согласие двух судей между собой — не доказательство качества: оба могут ошибаться одинаково. Но **расхождение** — дешёвый сигнал, что пример стоит показать человеку. Оценки порядковые (1–5), поэтому используем взвешенную каппу с квадратичными весами.

In [ ]:
if USE_LOCAL_JUDGE:
    verdicts = [parse_verdict(o) for o in generate("phi", [judge_prompt(t, "traj") for t in LS])]
    J["phi_traj"] = [v.score if v else np.nan for v in verdicts]
    print(f"phi_traj: не распарсилось {sum(v is None for v in verdicts)} из {len(verdicts)}")
    both = J[["qwen_traj", "phi_traj"]].dropna().astype(int)
    print(f"qwen vs phi: κ = {cohen_kappa_score(both.qwen_traj, both.phi_traj):.2f}, "
          f"взвешенная κ = {cohen_kappa_score(both.qwen_traj, both.phi_traj, weights='quadratic'):.2f}")
    display(agreement_with_human(J, ["qwen_traj", "phi_traj"]))
    J["судьи_расходятся"] = (J.qwen_traj - J.phi_traj).abs() >= 2
    display(J[J["судьи_расходятся"]])
else:
    print("нужен GPU")

### 4.3 Смещение по позиции

Pairwise-судья сравнивает два ответа на одну задачу: верный (по согласованной разметке `success` без ошибок) и неверный (`fail`). Каждую пару спрашиваем дважды, поменяв порядок. Если судья смотрит на содержание, победитель не зависит от порядка.

In [ ]:
PAIR_TMPL = """Ты — строгий асессор качества ответов службы поддержки интернет-магазина.
Клиент написал: {request}

Ответ A: {a}
Ответ B: {b}

Какой ответ лучше: точнее, честнее и полезнее для клиента? Кратко порассуждай, затем верни только JSON:
{{"reasoning": "...", "winner": "A" или "B"}}"""

good = H[(H.outcome == "success") & (H.error_category == "none")].index
bad = H[H.outcome == "fail"].index
PAIRS = [(g, b) for g in good for b in bad if TR[g]["task_id"] == TR[b]["task_id"]]
print(f"пар: {len(PAIRS)}")


def winner(text):
    m = re.search(r'winner"?\s*:\s*"?([AB])', text)
    return m.group(1) if m else None


if USE_LOCAL_JUDGE:
    def pair_prompt(x, y):
        return PAIR_TMPL.format(request=TASKS[TR[x]["task_id"]]["text"], a=final_answer(TR[x]), b=final_answer(TR[y]))
    ab = [winner(o) for o in generate("qwen", [pair_prompt(g, b) for g, b in PAIRS])]  # верный ответ на месте A
    ba = [winner(o) for o in generate("qwen", [pair_prompt(b, g) for g, b in PAIRS])]  # верный ответ на месте B
    P = pd.DataFrame({"pair": PAIRS, "good_first": ab, "good_second": ba}).dropna()
    P["consistent"] = (P.good_first == "A") == (P.good_second == "B")
    P["picks_good_both"] = (P.good_first == "A") & (P.good_second == "B")
    print(f"согласован при перестановке: {P.consistent.mean():.0%}")
    print(f"выбрал верный ответ в обоих порядках: {P.picks_good_both.mean():.0%}")
    all_verdicts = list(P.good_first) + list(P.good_second)
    print(f"доля вердиктов «A»: {np.mean([w == 'A' for w in all_verdicts]):.0%} (без смещения — 50 %)")
else:
    print("нужен GPU")

Если судья согласован при перестановке заметно реже чем в 100 % пар, вердикты по одному порядку использовать нельзя. Лечение из лекции: каждую пару в обоих порядках; несовпадение — ничья. Для ориентира: у GPT-4 на MT-Bench согласованность была около 65 % пар, у слабых судей ниже. Ваша маленькая модель — сравните.

### 4.4 Смещение по длине

Раздуваем финальные ответы водой без новых фактов и сравниваем оценки судьи по тексту до и после. Особенно интересно, что происходит с **неверными** ответами: делает ли вода ложь убедительнее?

In [ ]:
WATER_PREFIX = "Здравствуйте! Спасибо, что обратились в нашу службу поддержки, мы очень ценим ваше доверие. "
WATER_SUFFIX = (" Если у вас остались вопросы, обязательно пишите — мы всегда рады помочь и сделать ваш опыт покупок "
                "максимально комфортным. Хорошего дня!")

with_answer = [t for t in LS if t["steps"][-1]["tool"] == "finish"]
if USE_LOCAL_JUDGE:
    plain = [parse_verdict(o) for o in generate("qwen", [judge_prompt(t, "text") for t in with_answer])]
    padded = [parse_verdict(o) for o in generate("qwen", [judge_prompt(t, "text", WATER_PREFIX + final_answer(t) + WATER_SUFFIX)
                                                         for t in with_answer])]
    V = pd.DataFrame({"trace_id": [t["trace_id"] for t in with_answer],
                      "plain": [v.score if v else np.nan for v in plain],
                      "padded": [v.score if v else np.nan for v in padded]}).dropna()
    V["human"] = ADJ.set_index("trace_id").loc[V.trace_id, "outcome"].to_numpy()
    V["delta"] = V.padded - V.plain
    display(V.groupby("human")[["plain", "padded", "delta"]].mean().round(2))
else:
    print("нужен GPU")

Положительная `delta` — судья платит за длину. Лечение из лекции: критерий краткости в rubric или length-controlled сравнение.

**⭐ Задание (шум судьи):** выберите 5 трасс, прогоните `generate("qwen", ..., temperature=0.7, seed=s)` для `s` от 0 до 9 и посчитайте σ судьи по каждой трассе. Подставьте σ в калькулятор «Шум против эффекта» из лекции вместе с n и Δ из раздела 5: виден ли эффект сквозь шум?

**⭐ Задание (самопредпочтение):** здесь его не проверить — трассы написаны руками, а не моделью. Как бы вы поставили эксперимент, если бы агент работал на Qwen? Что делает `blind_label` из прошлогоднего семинара и почему его недостаточно?

## 5. v1 против v2: шум против эффекта

v1 — базовый промпт агента. v2 — промпт «сначала проверь заказ, будь проактивен». Вопрос из лекции: **стало ли лучше, и выкатываем ли?** Цепочка «гипотеза → вариант → набор → метрика → интервал → решение».

### 5.1 North-star: доля решённых задач

In [ ]:
EV = E[E.split == "visible"]  # решения принимаются на видимой части; отложенная ждёт финального замера
P = EV.groupby(["task_id", "version"]).success.mean().unstack()
d = (P.v2 - P.v1).to_numpy()
ci = bootstrap((d,), np.mean, n_resamples=10_000, method="percentile", random_state=0).confidence_interval

print(f"задач: {len(P)}; доля успеха: v1 = {P.v1.mean():.2f}, v2 = {P.v2.mean():.2f}")
print(f"разница (парная по задачам): {d.mean():+.2f}, 95% bootstrap-интервал [{ci.low:+.2f}; {ci.high:+.2f}]")

Bootstrap — парный: разница считается внутри каждой задачи, потом задачи пересэмплируются. Так задачи, которые одинаково трудны для обеих версий, не добавляют шума.

Если интервал накрывает ноль — на этом наборе версии неразличимы, сколько бы знаков после запятой ни было в средних. 9 задач видимой части — мало: см. 1.2.

### 5.2 По категориям и ограничивающие метрики

In [ ]:
by_cat = EV.pivot_table(index="category", columns="version", values="success", aggfunc="mean").round(2)
by_cat["Δ"] = by_cat.v2 - by_cat.v1
display(by_cat.sort_values("Δ"))

by_ver = EV.groupby("version")
succ_v = EV.groupby(["task_id", "version"]).success.sum().unstack()
pd.DataFrame({
    "успех": by_ver.success.mean(),
    "pass^5": {v: np.mean([env.pass_hat_k(5, c, 5) for c in succ_v[v]]) for v in ["v1", "v2"]},
    "побочные действия": by_ver.side_effects.sum(),
    "трасс с циклом": by_ver.cycles.apply(lambda x: int((x > 0).sum())),
    "шагов в среднем": by_ver.steps.mean(),
}).round(2)

### 5.3 Решение

Соберите ответ сами, но вот что должно в нём быть:

- **North-star** выросла, но интервал на 9 задачах, скорее всего, накрывает ноль. Как самостоятельный аргумент этого мало.
- **pass^5** вырос заметно: v2 надёжнее на повторных прогонах.
- **Ограничивающая метрика «побочные действия»** у v2 хуже: в T06 агент «проактивно» оформляет возврат, когда просили отмену. По правилу из лекции ограничивающая метрика не должна ухудшаться, поэтому v2 в таком виде не выкатывается. Следующий эксперимент меняет **одно**: запрет на действия без явной просьбы в промпте v2.
- Отложенная часть (T09–T11) в решении не участвует. Что v2 делает там, станет известно только на финальном замере — и если там найдётся регрессия, это повод пополнить видимую часть похожими задачами, а не подглядывать в отложенную.

Ровно так выглядит раздел «план усложнений» в design doc: что пробуем, по какой метрике решаем, при каком условии переходим дальше.

## 6. Регрессионный тест и бюджет оценивания

### 6.1 Регрессия как тест

Каждое изменение промпта, модели или кода вокруг модели прогоняет набор. Порог на метрику; падение теста блокирует мерж. В проекте это DeepEval, promptfoo, Inspect или обычный pytest — механика одна. Тест запускается на **видимой** части набора.

In [ ]:
%%writefile test_shop_regression.py
import os
import sys

sys.path.insert(0, os.environ["SHOP_DIR"])
import shop_env as env

BASELINE, CANDIDATE = "v1", os.environ.get("CANDIDATE", "v2")
TOLERANCE = 0.05
TASKS = env.load_tasks()


def traces(version):
    return [t for t in env.load_traces()
            if t["agent_version"] == version and TASKS[t["task_id"]]["split"] == "visible"]


def success_rate(version):
    tr = traces(version)
    return sum(env.task_success(TASKS[t["task_id"]], t["steps"]) for t in tr) / len(tr)


def side_effects(version):
    n = 0
    for t in traces(version):
        task = TASKS[t["task_id"]]
        _, log = env.replay(task, t["steps"])
        n += sum(1 for r in log if r["changed"] and r["key"] not in task["golden"]["nodes"])
    return n


def test_success_not_worse():
    assert success_rate(CANDIDATE) >= success_rate(BASELINE) - TOLERANCE


def test_no_side_effects():
    assert side_effects(CANDIDATE) == 0, f"побочных действий: {side_effects(CANDIDATE)}"

In [ ]:
import os
os.environ["SHOP_DIR"] = str(SHOP.resolve())
!python -m pytest -q test_shop_regression.py

Первый тест проходит, второй падает на побочных действиях v2 — ровно то решение, к которому мы пришли в 5.3, только автоматически и на каждом PR.

### 6.2 Бюджет оценивания

Из лекции: набор × метрики-судьи × повторы × прогоны; для агентов ещё × k прогонов на задачу ради pass^k. Детерминированные проверки в счёт не входят. Если судья запускался в разделе 4, время одного вызова на T4 подставляется из замера.

In [ ]:
def eval_budget(n_examples, judge_metrics, repeats=1, runs_per_week=5, k_agent=1, sec_per_call=None, price_per_call=None):
    calls = n_examples * judge_metrics * repeats * runs_per_week * k_agent * 4.3  # в месяц
    out = {"вызовов судьи в месяц": int(calls),
           "прогонов агента в месяц": int(n_examples * k_agent * runs_per_week * 4.3)}
    if sec_per_call:
        out["GPU-часов в месяц"] = round(calls * sec_per_call / 3600, 1)
    if price_per_call:
        out["стоимость в месяц"] = round(calls * price_per_call, 2)
    return out


sec = JUDGE_STATS["seconds"] / JUDGE_STATS["calls"] if JUDGE_STATS["calls"] else None
if sec:
    print(f"замер: {JUDGE_STATS['calls']} вызовов, {sec:.2f} с на вызов (батч 8)")
eval_budget(n_examples=100, judge_metrics=2, repeats=1, runs_per_week=5, k_agent=5, sec_per_call=sec)

Подставьте числа своего проекта. k = 5 ради pass^k умножает всё в пять раз, и это обычно главная статья. Её уменьшают так: детерминированные проверки вместо судьи там, где это возможно, судья — только на финальный текст, k прогонов — только на подмножестве самых рискованных категорий.

## 7. Что из этого идёт в design doc (раздел 7, КТ2)

| Пункт | Где на семинаре |
|---|---|
| Категории задач, выведенные из контракта; сколько в каждой и почему; веса трафика и риска | 1.1, 1.2 |
| Запись эталона: состояние, текст, граф; для агентов — эталонные графы (DAG) | 1.3 |
| Тест эталона | 1.4 |
| Источники примеров; чем они отличаются от тех, на которых система разрабатывается | 1.5 |
| Видимая и отложенная части, версия набора | 1.6 |
| Инструкция разметчика и таксономия ошибок; двойная разметка на подвыборке; согласие (потолок метрики) | 2.2–2.5 |
| Что проверяется состоянием, что графом, что судьёй; у каждой метрики — пункт контракта | 1.1, 3 |
| Если есть судья: rubric с якорями, каппа против разметки, какие смещения проверены | 4 |
| k для pass^k | 3.6 |
| Решение по эксперименту: north-star, интервал, ограничивающие метрики | 5 |
| Регрессионный тест: порог, когда запускается | 6.1 |
| Бюджет оценивания на месяц | 6.2 |

**⭐ Задания, собранные вместе:**

1. Перефразы шаблонных задач локальной моделью и проверка, что смысл не поменялся (1.5).
2. Три задачи новой категории с эталоном и тестом эталона (1.6).
3. Правило для самой спорной пары категорий и переразметка (2.5).
4. Сделайте `get_order` необязательным узлом в T01 и T02. Как меняются метрики графа и согласие автоматики с человеком? Кто прав — эталон или разметчик?
5. Шум судьи и калькулятор «Шум против эффекта» (4.4).
6. Добавьте в `RUBRIC` критерий краткости и повторите тест на длину (4.4).

## 8. Дополнительные материалы

**Судьи и их смещения**
- Zheng et al., 2023 — [Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena](https://arxiv.org/abs/2306.05685): позиция, длина, самопредпочтение.
- Wang et al., 2023 — [Large Language Models are not Fair Evaluators](https://arxiv.org/abs/2305.17926): позиционное смещение и калибровка перестановкой.
- Liu et al., 2023 — [G-Eval](https://arxiv.org/abs/2303.16634): rubric и рассуждение до оценки.
- Kim et al., 2024 — [Prometheus 2](https://arxiv.org/abs/2405.01535): открытая специализированная модель-судья.
- Shankar et al., 2024 — [Who Validates the Validators?](https://arxiv.org/abs/2404.12272): критерии дрейфуют по мере разметки.

**Оценка агентов**
- Yao et al., 2024 — [τ-bench](https://arxiv.org/abs/2406.12045): состояние базы после диалога и pass^k.
- Qiao et al., 2024 — [WorfBench](https://arxiv.org/abs/2410.07869): эталонные графы с несколькими правильными путями.
- Zhuge et al., 2024 — [Agent-as-a-Judge](https://arxiv.org/abs/2410.10934): требования задачи как DAG, проверка по артефактам.
- Zhang et al., 2025 — [Which Agent Causes Task Failures and When?](https://arxiv.org/abs/2505.00212) (Who&When): поиск шага и агента, где всё сломалось.

**Что не писать руками**
- каппа: `sklearn.metrics.cohen_kappa_score`, `statsmodels.stats.inter_rater.fleiss_kappa`; bootstrap: `scipy.stats.bootstrap`; графы: `networkx`;
- регрессионные прогоны: DeepEval, promptfoo, Inspect; трейсы и датасеты: Langfuse.